# Notebook 1 — Data Ingestion
**Deadline:** Mar 1 (Weeks 1–6)  
**Goal:** Load GEFS + ERA5 weather data and EAGLE-I outage data — all free, no credentials required.

---

## Data Sources

| Dataset | Format | Access | Cost |
|---|---|---|---|
| **GEFS Analysis** (wind, precip, RH) | **Zarr v3** via dynamical.org | `https://data.dynamical.org/noaa/gefs/analysis/latest.zarr` | Free, no credentials |
| **ERA5 Reanalysis** (CAPE, soil moisture, wind gust) | **Zarr** via ARCO-ERA5 (Google Cloud) | `gs://gcp-public-data-arco-era5/...` | Free, anonymous |
| **EAGLE-I Outages** | CSV | DOE figshare DOI | Free |
| **US County Shapefile** | Shapefile | US Census TIGER | Free |

### Why two weather sources?
- **dynamical.org GEFS** provides wind (u/v 10m), precipitation, precipitable water, RH, and temperature in clean Zarr v3 / IceChunk format covering 2000–present.
- **ARCO-ERA5** fills the critical gaps: **CAPE, CIN, soil moisture, and wind gusts** — variables not available in the dynamical.org GEFS analysis but essential to the Hill et al. / Saki et al. methodology.

> ⚠️ **IceChunk note:** dynamical.org's IceChunk URLs are pre-release. Subscribe to their newsletter for any URL changes when IceChunk 2.0 is released.

---

In [1]:
# ============================================================
# INSTALL ZARR / ICECHUNK PACKAGES
# Run once, then restart kernel
# ============================================================
import subprocess, sys

zarr_packages = [
    "zarr>=3.0.8",
    "xarray>=2025.1.2",
    "icechunk",
    "gcsfs",
    "fsspec",
    "aiohttp",
    "dask",
]
for pkg in zarr_packages:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True, text=True)
    print(f"  {'✓' if result.returncode == 0 else '✗'}  {pkg}")
print("\nDone. Restart kernel if first run.")

  ✓  zarr>=3.0.8
  ✓  xarray>=2025.1.2
  ✓  icechunk
  ✓  gcsfs
  ✓  fsspec
  ✓  aiohttp
  ✓  dask

Done. Restart kernel if first run.


In [ ]:
# ============================================================
# IMPORTS & CONFIG
# ============================================================
import os, json, zipfile, warnings
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import gcsfs
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ← EDIT THIS to your project folder
PROJECT_ROOT  = Path("/data/keeling/a/tob3/Capstone2026/Testingthings")

DATA_RAW      = PROJECT_ROOT / "data" / "raw"
DATA_PROC     = PROJECT_ROOT / "data" / "processed"
EAGLEI_DIR    = DATA_RAW / "eaglei"
COUNTY_SHP    = DATA_RAW / "counties"

REGION5_FIPS  = ["17", "18", "26", "27", "39", "55"]
DOMAIN_LAT    = (36.5, 49.5)
DOMAIN_LON    = (-97.5, -80.0)
TRAIN_YEARS   = [2021, 2022]
TEST_YEARS    = [2023]
OUTAGE_THRESHOLD = 0.01

for d in [DATA_RAW, DATA_PROC, EAGLEI_DIR, COUNTY_SHP]:
    d.mkdir(parents=True, exist_ok=True)

print("Config loaded. Project root:", PROJECT_ROOT)

Config loaded. Project root: /data/keeling/a/tob3/Testingthings


---
## Part 1 — US County Shapefile

In [3]:
COUNTY_URL = "https://www2.census.gov/geo/tiger/TIGER2022/COUNTY/tl_2022_us_county.zip"
COUNTY_ZIP = COUNTY_SHP / "tl_2022_us_county.zip"

def download_file(url, dest, chunk_size=1 << 20):
    if dest.exists():
        print(f"  Already exists: {dest.name}"); return
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
        for chunk in r.iter_content(chunk_size):
            f.write(chunk); bar.update(len(chunk))

download_file(COUNTY_URL, COUNTY_ZIP)
with zipfile.ZipFile(COUNTY_ZIP) as zf:
    zf.extractall(COUNTY_SHP)

counties_all = gpd.read_file(COUNTY_SHP / "tl_2022_us_county.shp")
counties_r5  = counties_all[counties_all["STATEFP"].isin(REGION5_FIPS)].copy().to_crs(epsg=4326)
counties_r5["centroid_lon"] = counties_r5.geometry.centroid.x
counties_r5["centroid_lat"] = counties_r5.geometry.centroid.y
counties_r5.to_file(COUNTY_SHP / "counties_region5.gpkg", driver="GPKG")
print(f"FEMA Region 5 counties: {len(counties_r5)}")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

---
## Part 2 — EAGLE-I Outage Data

In [ ]:
# DOI: https://doi.org/10.6084/m9.figshare.24237376
EAGLEI_URLS = {
    2021: "https://figshare.com/ndownloader/files/43009749",
    2022: "https://figshare.com/ndownloader/files/43009752",
    2023: "https://figshare.com/ndownloader/files/43009755",
}

eaglei_frames = []
for yr, url in EAGLEI_URLS.items():
    dest = EAGLEI_DIR / f"eaglei_outages_{yr}.csv"
    download_file(url, dest)
    df = pd.read_csv(dest, parse_dates=["run_start_time"], low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    df["fips_str"]   = df["fips_code"].astype(str).str.zfill(5)
    df["state_fips"] = df["fips_str"].str[:2]
    df = df[df["state_fips"].isin(REGION5_FIPS)].copy()
    df["outage_frac"]  = (df["customers_out"] / df["total_customers"]).clip(0, 1)
    df["outage_event"] = (df["outage_frac"] >= OUTAGE_THRESHOLD).astype(int)
    eaglei_frames.append(df)
    print(f"{yr}: {len(df):,} rows | outage events: {df['outage_event'].sum():,} ({df['outage_event'].mean()*100:.1f}%)")

eaglei_all = pd.concat(eaglei_frames, ignore_index=True).sort_values("run_start_time")
eaglei_all.to_parquet(DATA_RAW / "eaglei_region5_all.parquet", index=False)
print(f"\nSaved {len(eaglei_all):,} total records.")

---
## Part 3 — GEFS Analysis via dynamical.org (Zarr v3, free)

**No AWS account. No credentials. One line.**

Variables available here:
`wind_u_10m`, `wind_v_10m`, `precipitation_surface`, `precipitable_water_atmosphere`, `relative_humidity_2m` *(2021+)*, `temperature_2m`, `pressure_surface`

In [ ]:
# ============================================================
# OPEN GEFS ZARR — lazy, no download until .compute()
# xarray>=2025.1.2 and zarr>=3.0.8 required for Zarr v3
# ============================================================

GEFS_ZARR_URL = "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr"

print("Opening GEFS Zarr (lazy)...")
ds_gefs = xr.open_zarr(GEFS_ZARR_URL, chunks="auto")

print(f"Dimensions: {dict(ds_gefs.dims)}")
print(f"Time range: {str(ds_gefs.time.values[0])[:19]} → {str(ds_gefs.time.values[-1])[:19]}")
print(f"Variables: {list(ds_gefs.data_vars)}")

# Subset to FEMA Region 5 (lazy)
GEFS_VARS = ["wind_u_10m", "wind_v_10m", "precipitation_surface",
             "precipitable_water_atmosphere", "relative_humidity_2m", "temperature_2m"]

ds_gefs_r5 = ds_gefs[[v for v in GEFS_VARS if v in ds_gefs]].sel(
    latitude=slice(DOMAIN_LAT[1], DOMAIN_LAT[0]),
    longitude=slice(DOMAIN_LON[0], DOMAIN_LON[1]),
)
print(f"\nRegion 5 subset: {dict(ds_gefs_r5.dims)}")

In [ ]:
# Sanity check: load one time step (small download)
print("Loading one timestep...")
test = ds_gefs_r5[["wind_u_10m", "wind_v_10m"]].sel(time="2021-06-01T00", method="nearest").compute()
wind_spd = np.sqrt(test["wind_u_10m"]**2 + test["wind_v_10m"]**2)

fig, ax = plt.subplots(figsize=(8, 4))
wind_spd.plot(ax=ax, cmap="viridis")
ax.set_title("GEFS 10m Wind Speed — 2021-06-01 00Z")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "01_gefs_sanity.png", dpi=100)
plt.show()
print(f"Range: {float(wind_spd.min()):.2f} – {float(wind_spd.max()):.2f} m/s ✓")

---
## Part 4 — ERA5 via ARCO-ERA5 (Google Cloud, free, anonymous Zarr)

Fills the GEFS gaps — **CAPE, CIN, soil moisture (`swvl1`), wind gust (`fg10`)**.

Source: https://github.com/google-research/arco-era5

In [ ]:
# ============================================================
# OPEN ARCO-ERA5 — single-level reanalysis, anonymous access
# ============================================================

ERA5_SL_URL = "gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr-v2"

print("Opening ARCO-ERA5 (lazy, anonymous)...")
fs_gcs  = gcsfs.GCSFileSystem(token="anon")
ds_era5 = xr.open_zarr(fs_gcs.get_mapper(ERA5_SL_URL), chunks="auto")

print(f"Dimensions: {dict(ds_era5.dims)}")
print(f"Time range: {str(ds_era5.time.values[0])[:19]} → {str(ds_era5.time.values[-1])[:19]}")

# Target ERA5 variables
ERA5_TARGET = ["cape", "cin", "swvl1", "fg10"]
ERA5_AVAIL  = [v for v in ERA5_TARGET if v in ds_era5]
ERA5_MISS   = [v for v in ERA5_TARGET if v not in ds_era5]

print(f"\nTarget variables: {ERA5_TARGET}")
print(f"  Found: {ERA5_AVAIL}")
if ERA5_MISS:
    print(f"  Missing: {ERA5_MISS} — check full_37-1h-0p25deg store or variable naming")

# Subset to Region 5
ds_era5_r5 = ds_era5[ERA5_AVAIL].sel(
    latitude=slice(DOMAIN_LAT[1], DOMAIN_LAT[0]),
    longitude=slice(DOMAIN_LON[0], DOMAIN_LON[1]),
)
print(f"\nRegion 5 subset: {dict(ds_era5_r5.dims)}")

In [ ]:
# ERA5 sanity check — CAPE on a convective day
if "cape" in ds_era5_r5:
    print("Loading ERA5 CAPE for 2021-06-01 00Z...")
    cape = ds_era5_r5["cape"].sel(time="2021-06-01T00", method="nearest").compute()
    fig, ax = plt.subplots(figsize=(8, 4))
    cape.plot(ax=ax, cmap="hot_r", vmin=0, vmax=3000)
    ax.set_title("ERA5 Surface CAPE — 2021-06-01 00Z (FEMA Region 5)")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "outputs" / "01_era5_cape_sanity.png", dpi=100)
    plt.show()
    print(f"CAPE range: {float(cape.min()):.0f} – {float(cape.max()):.0f} J/kg ✓")
else:
    print("CAPE not found in this store — check ERA5 variable names above")

---
## Part 5 — Combined Feature Schema

In [ ]:
# ============================================================
# UNIFIED FEATURE SOURCE MAP
# Maps project internal names → (source, zarr_var, description)
# ============================================================

FEATURE_SOURCE_MAP = {
    # GEFS (dynamical.org Zarr v3)
    "ugrd_10m":          ("gefs", "wind_u_10m",                    "10m U-wind (m/s)"),
    "vgrd_10m":          ("gefs", "wind_v_10m",                    "10m V-wind (m/s)"),
    "apcp_sfc":          ("gefs", "precipitation_surface",         "Precip rate (kg/m2/s)"),
    "pwat_clm":          ("gefs", "precipitable_water_atmosphere", "Precipitable water (kg/m2)"),
    "rh_2m":             ("gefs", "relative_humidity_2m",          "2m RH (%) [2021+]"),
    "tmp_2m":            ("gefs", "temperature_2m",                "2m temperature (C)"),
    # ERA5 (ARCO-ERA5, Google Cloud)
    "cape_sfc":          ("era5", "cape",   "Surface CAPE (J/kg)"),
    "cin_sfc":           ("era5", "cin",    "Surface CIN (J/kg)"),
    "soilw_0_7":         ("era5", "swvl1",  "Soil moisture 0-7cm (m3/m3)"),
    "gust_sfc":          ("era5", "fg10",   "Max 10m wind gust (m/s)"),
    # Derived (computed in Notebook 2)
    "wind_speed_10m":    ("derived", None, "sqrt(u^2 + v^2)"),
    "wind_gust_x_soilw": ("derived", None, "gust * soilw — Saki et al. 2025"),
    "wind_gust_x_apcp":  ("derived", None, "gust * precip — compound hazard"),
    "cape_x_rh":         ("derived", None, "CAPE * RH"),
    # Static (Notebook 2)
    "forest_fraction":   ("static", None, "NLCD canopy cover fraction"),
    "pop_density":       ("static", None, "Population density (Census)"),
}

# Save for Notebooks 2-4
schema = {
    "feature_cols": list(FEATURE_SOURCE_MAP.keys()),
    "label_col": "outage_event",
    "sources": {k: {"source": v[0], "zarr_var": v[1], "desc": v[2]}
                for k, v in FEATURE_SOURCE_MAP.items()},
    "zarr_urls": {
        "gefs": GEFS_ZARR_URL,
        "era5_sl": ERA5_SL_URL,
    }
}
DATA_PROC.mkdir(parents=True, exist_ok=True)
with open(DATA_PROC / "feature_schema.json", "w") as f:
    json.dump(schema, f, indent=2)

gefs_n   = sum(1 for v in FEATURE_SOURCE_MAP.values() if v[0] == "gefs")
era5_n   = sum(1 for v in FEATURE_SOURCE_MAP.values() if v[0] == "era5")
deriv_n  = sum(1 for v in FEATURE_SOURCE_MAP.values() if v[0] == "derived")
static_n = sum(1 for v in FEATURE_SOURCE_MAP.values() if v[0] == "static")

print(f"Feature schema saved ({gefs_n} GEFS + {era5_n} ERA5 + {deriv_n} derived + {static_n} static = {len(FEATURE_SOURCE_MAP)} total)")
print("\n✅ Notebook 1 complete — all data sources verified, no credentials required!")
print("   Ready for 02_geospatial_integration.ipynb")